In [3]:
import xarray as xr
import numpy as np
from scipy.spatial import cKDTree

def get_cellid(G, nc_filepath, lat_name="lat", lon_name="lon"):
    """
    Ordnet jeder Edge eines OSMnx-Graphen eine Rasterzelle (cell_id) zu.

    Parameter
    ----------
    G : networkx.MultiDiGraph
        OSMnx-Graph
    nc_filepath: str
        Pfad zur NetCDF-Datei
    lat_name : str
        Name der Latitude-Variable im Dataset
    lon_name : str
        Name der Longitude-Variable im Dataset

    Returns
    -------
    G : networkx.MultiDiGraph
        Graph mit neuen Edge-Attributen:
        - cell_i
        - cell_j
        - cell_id
    """

    # ═══════════════════════════════════════════════════════════════
    # 1. NetCDF laden
    # ═══════════════════════════════════════════════════════════════
    ds = xr.open_dataset(nc_filepath)

    lat = ds[lat_name].values
    lon = ds[lon_name].values

    # KD-Tree
    points = np.column_stack([lon.ravel(), lat.ravel()])
    tree = cKDTree(points)

    # Grid-Indizes
    flat_idx = np.arange(len(points))
    i_all, j_all = np.unravel_index(flat_idx, lat.shape)

    # ═══════════════════════════════════════════════════════════════
    # 2. Edges extrahieren
    # ═══════════════════════════════════════════════════════════════
    edges = list(G.edges(keys=True, data=True))

    valid_edges = [(u, v, k) for (u, v, k, d) in edges if "geometry" in d]
    geoms = [d["geometry"] for (_, _, _, d) in edges if "geometry" in d]

    if len(geoms) == 0:
        raise ValueError("Keine Edge-Geometrien im Graph gefunden.")

    # ═══════════════════════════════════════════════════════════════
    # 3. Mittelpunkte berechnen
    # ═══════════════════════════════════════════════════════════════
    midpoints = np.array([
        geom.interpolate(0.5, normalized=True).coords[0]
        for geom in geoms
    ])

    # ═══════════════════════════════════════════════════════════════
    # 4. KD-Tree Query
    # ═══════════════════════════════════════════════════════════════
    _, idx = tree.query(midpoints)

    cell_i = i_all[idx]
    cell_j = j_all[idx]

    # Vektorisierte cell_id
    cell_ids = np.char.add(cell_i.astype(str), "_")
    cell_ids = np.char.add(cell_ids, cell_j.astype(str))

    # ═══════════════════════════════════════════════════════════════
    # 5. Zurück in Graph schreiben
    # ═══════════════════════════════════════════════════════════════
    for (u, v, k), i, j, cid in zip(valid_edges, cell_i, cell_j, cell_ids):
        G.edges[u, v, k]["cell_i"] = int(i)
        G.edges[u, v, k]["cell_j"] = int(j)
        G.edges[u, v, k]["cell_id"] = str(cid)

    return G

In [4]:
import osmnx as ox

G = ox.graph_from_place("Zürich, Switzerland", network_type="drive")
NC = 'RAINY_DATA_DEMO_icon_ch1_TOT_PREC_all_lead_times.nc'


In [5]:
G = get_cellid(G, NC)

# Test
u, v, k = list(G.edges(keys=True))[3]
print(G.edges[u, v, k])

{'osmid': [374688188, 603655389, 374688190], 'highway': 'tertiary', 'lanes': '3', 'maxspeed': '50', 'name': 'Hohlstrasse', 'oneway': False, 'reversed': True, 'length': np.float64(277.0583820264522), 'geometry': <LINESTRING (8.508 47.385, 8.508 47.385, 8.508 47.385, 8.508 47.385, 8.508 4...>, 'cell_i': 182, 'cell_j': 210, 'cell_id': '182_210'}


In [6]:
import xarray as xr

ds = xr.open_dataset("RAINY_DATA_DEMO_icon_ch1_TOT_PREC_all_lead_times.nc")

In [7]:
for u, v, key, data in G.edges(keys=True, data=True):
    print(u, v, key)
    print(data)

    break  # nur erste Edge anschauen

453768 11767444238 0
{'osmid': [1443804326, 9386989, 284193360, 438520592, 134223699, 438521946], 'highway': ['primary', 'motorway'], 'lanes': ['3', '2'], 'maxspeed': '60', 'oneway': True, 'ref': ['1;3', 'A1H'], 'reversed': False, 'length': np.float64(741.1529450751349), 'name': 'Bernerstrasse Süd', 'geometry': <LINESTRING (8.486 47.395, 8.488 47.394, 8.489 47.394, 8.49 47.394, 8.491 47...>, 'cell_i': 183, 'cell_j': 210, 'cell_id': '183_210'}


In [8]:
def get_forecast(G, ds, u, v, k,
                      lead_idx=0,
                      eps_idx=0,
                      ref_time_idx=0,
                      var_name="TOT_PREC"):
    """
    Extrahiert den Forecast-Wert (z.B. Niederschlag) für eine einzelne Edge im OSMnx-Graphen.

    Diese Funktion verbindet:
    - OSMnx-Graph (Straßennetz)
    - Rasterdaten aus einem xarray NetCDF Dataset
    - vorher berechnetes Mapping (cell_i, cell_j)

    Parameter
    ----------
    G : networkx.MultiDiGraph
        OSMnx-Graph, bei dem jede Edge bereits eine Rasterzuordnung besitzt
        ('cell_i', 'cell_j')

    ds : xarray.Dataset
        Geladenes NetCDF-Dataset mit Wetterdaten

    u, v, k : int
        Eindeutige Edge-Identifikation im Graph:
        - u = Startknoten
        - v = Zielknoten
        - k = Edge-Key (bei parallelen Straßen)

    lead_idx : int
        Index der Vorhersagezeit (Lead Time), z.B. 0 = erste Stunde

    eps_idx : int
        Ensemble-Mitglied (falls vorhanden)

    ref_time_idx : int
        Startzeit-Index des Modelllaufs

    var_name : str
        Name der Wettervariablen im Dataset (z.B. "TOT_PREC")

    Returns
    -------
    float
        Wetterwert (z.B. Niederschlag) für diese Edge und Zeitkombination
    """

    # ═══════════════════════════════════════════════
    # 1. Edge aus dem Graphen holen
    # ═══════════════════════════════════════════════
    edge = G.edges[u, v, k]

    # Sicherheitscheck: wurde die Rasterzuordnung bereits berechnet?
    if "cell_i" not in edge or "cell_j" not in edge:
        raise ValueError(
            "Edge hat keine cell_i/cell_j. Bitte zuerst get_cellid() ausführen."
        )

    # Rasterposition der Straße im Wettergitter
    i = edge["cell_i"]  # y-Koordinate im Raster
    j = edge["cell_j"]  # x-Koordinate im Raster

    # ═══════════════════════════════════════════════
    # 2. Zugriff auf das Wetter-Dataset
    # ═══════════════════════════════════════════════
    data = ds[var_name]

    # ═══════════════════════════════════════════════
    # 3. Sauberes xarray Indexing über Dimensionen
    #    (robust, unabhängig von interner Reihenfolge)
    # ═══════════════════════════════════════════════
    value = data.isel(
        eps=eps_idx,
        ref_time=ref_time_idx,
        lead_time=lead_idx,
        y=i,
        x=j
    ).values

    # ═══════════════════════════════════════════════
    # 4. Rückgabe als Python float
    # ═══════════════════════════════════════════════
    return float(value)

In [9]:
u, v, k = list(G.edges(keys=True))[0]

rain = get_forecast(
    G,
    ds,
    u, v, k,
    lead_idx=30
)

print(rain)

0.12948113546281848


In [10]:
def get_forecast_all(G, ds, u, v, k,
                     eps_idx=0,
                     ref_time_idx=0,
                     var_name="TOT_PREC"):
    """
    Gibt die komplette Zeitreihe (alle lead_times) für eine Edge zurück.

    Parameter
    ----------
    G : networkx.MultiDiGraph
        OSMnx-Graph mit 'cell_i' und 'cell_j'
    ds : xarray.Dataset
        Wetter-Dataset
    u, v, k : int
        Edge-Identifier im Graph
    eps_idx : int
        Ensemble-Mitglied
    ref_time_idx : int
        Modelllauf
    var_name : str
        Name der Wettervariable

    Returns
    -------
    np.ndarray
        1D Array: (lead_time,) → Zeitreihe der Regenwerte
    """

    # ═══════════════════════════════════════════════
    # 1. Edge holen
    # ═══════════════════════════════════════════════
    edge = G.edges[u, v, k]

    if "cell_i" not in edge or "cell_j" not in edge:
        raise ValueError("Edge hat keine cell_i/cell_j → zuerst get_cellid() ausführen")

    i = edge["cell_i"]
    j = edge["cell_j"]

    # ═══════════════════════════════════════════════
    # 2. Dataset auswählen
    # ═══════════════════════════════════════════════
    data = ds[var_name]

    # ═══════════════════════════════════════════════
    # 3. Alle lead_times extrahieren
    #    → Ergebnis: (lead_time,)
    # ═══════════════════════════════════════════════
    series = data.isel(
        eps=eps_idx,
        ref_time=ref_time_idx,
        y=i,
        x=j
    ).values

    return series

In [ ]:
u, v, k = list(G.edges(keys=True))[0]

rain = get_forecast_all(
    G,
    ds,
    u, v, k,
    
)

print(u,v,k)
print(rain)

453768 11767444238 0
[0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.12948114 0.12948114 0.12948114 0.12948114
 0.12948114 0.12948114 0.12948114 0.12948114 0.12948114 0.12948114
 0.12948114 0.12948114 0.12948114 0.12948114]


In [12]:
import numpy as np
import pandas as pd

def grid_to_table(ds, var_name="TOT_PREC", lead_idx=0, eps_idx=0, ref_time_idx=0):
    """
    Konvertiert ein 2D Raster (y, x) aus einem xarray Dataset
    in eine tabellarische Form mit cell_id.

    Parameter
    ----------
    ds : xarray.Dataset
        Wetter-Dataset
    var_name : str
        Name der Variable
    lead_idx : int
        Forecast-Zeitindex
    eps_idx : int
        Ensemble Index
    ref_time_idx : int
        Referenzzeit Index

    Returns
    -------
    pandas.DataFrame
        Tabelle mit:
        - cell_id
        - y
        - x
        - value
    """

    data = ds[var_name]

    # ═══════════════════════════════════════════════
    # 1. 2D Slice extrahieren
    # ═══════════════════════════════════════════════
    grid = data.isel(
        eps=eps_idx,
        ref_time=ref_time_idx,
        lead_time=lead_idx
    ).values  # shape (y, x)

    ny, nx = grid.shape

    # ═══════════════════════════════════════════════
    # 2. Indizes erzeugen
    # ═══════════════════════════════════════════════
    y_idx, x_idx = np.meshgrid(
        np.arange(ny),
        np.arange(nx),
        indexing="ij"
    )

    y_flat = y_idx.ravel()
    x_flat = x_idx.ravel()
    values = grid.ravel()

    # ═══════════════════════════════════════════════
    # 3. cell_id erzeugen
    # ═══════════════════════════════════════════════
    cell_id = np.char.add(y_flat.astype(str), "_")
    cell_id = np.char.add(cell_id, x_flat.astype(str))

    # ═══════════════════════════════════════════════
    # 4. DataFrame bauen
    # ═══════════════════════════════════════════════
    df = pd.DataFrame({
        "cell_id": cell_id,
        "y": y_flat,
        "x": x_flat,
        "value": values
    })

    return df

In [18]:
df = grid_to_table(ds, var_name="TOT_PREC", lead_idx=3)

print(df.head())
print(df[df['cell_id'] == '160_100'])

  cell_id  y  x  value
0     0_0  0  0    NaN
1     0_1  0  1    NaN
2     0_2  0  2    NaN
3     0_3  0  3    NaN
4     0_4  0  4    NaN
       cell_id    y    x  value
68740  160_100  160  100    0.0
